In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [5]:

# 1. Load the dataset 
# Often this dataset uses ';' as a separator and ',' as a decimal point
df = pd.read_csv('../data/AirQualityUCI.csv', sep=';', decimal=',')

# 2. Clean up -200 placeholders and empty columns/rows
df.replace(-200, np.nan, inplace=True)
df.dropna(subset=['C6H6(GT)'], inplace=True) # Drop rows where target is missing
df = df.iloc[:, :15] # The UCI version often has trailing empty columns

print(df.head())


         Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
0  10/03/2004  18.00.00     2.6       1360.0     150.0      11.9   
1  10/03/2004  19.00.00     2.0       1292.0     112.0       9.4   
2  10/03/2004  20.00.00     2.2       1402.0      88.0       9.0   
3  10/03/2004  21.00.00     2.2       1376.0      80.0       9.2   
4  10/03/2004  22.00.00     1.6       1272.0      51.0       6.5   

   PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)  \
0         1046.0    166.0        1056.0    113.0        1692.0       1268.0   
1          955.0    103.0        1174.0     92.0        1559.0        972.0   
2          939.0    131.0        1140.0    114.0        1555.0       1074.0   
3          948.0    172.0        1092.0    122.0        1584.0       1203.0   
4          836.0    131.0        1205.0    116.0        1490.0       1110.0   

      T    RH      AH  
0  13.6  48.9  0.7578  
1  13.3  47.7  0.7255  
2  11.9  54.0  0.7502  
3  11.0  60.0  0.786

In [6]:
print(df.isna().sum())

Date                0
Time                0
CO(GT)           1647
PT08.S1(CO)         0
NMHC(GT)         8104
C6H6(GT)            0
PT08.S2(NMHC)       0
NOx(GT)          1595
PT08.S3(NOx)        0
NO2(GT)          1598
PT08.S4(NO2)        0
PT08.S5(O3)         0
T                   0
RH                  0
AH                  0
dtype: int64


In [7]:


# 3. Feature Engineering: Date and Time
# Convert to datetime objects to extract numerical features
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

# Extract hour from the 'Time' string (e.g., "18.00.00" -> 18)
df['Hour'] = df['Time'].str.split('.').str[0].astype(int)

In [8]:


# 4. Drop original Date/Time and handle remaining NaNs
df = df.drop(columns=['Date', 'Time'])
df = df.fillna(df.median()) # KNN cannot handle NaNs

In [9]:
print(df.isna().sum())

CO(GT)           0
PT08.S1(CO)      0
NMHC(GT)         0
C6H6(GT)         0
PT08.S2(NMHC)    0
NOx(GT)          0
PT08.S3(NOx)     0
NO2(GT)          0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
Month            0
DayOfWeek        0
Hour             0
dtype: int64


In [10]:



# 5. Split Features (X) and Target (y)
# Targeting Benzene (C6H6) concentration
X = df.drop(columns=['C6H6(GT)'])
y = df['C6H6(GT)']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Scaling (MANDATORY for KNN)
# KNN uses Euclidean Distance:
# $$d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 8. Reconstruct and Save
train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
train_final['y'] = y_train.values

test_final = pd.DataFrame(X_test_scaled, columns=X.columns)
test_final['y'] = y_test.values

train_final.to_csv('../data/air_quality_preprocessed_train.csv', index=False)
test_final.to_csv('../data/air_quality_preprocessed_test.csv', index=False)

print("Preprocessing complete!")
print(f"Features used: {list(X.columns)}")

Preprocessing complete!
Features used: ['CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'Month', 'DayOfWeek', 'Hour']
